In [1]:
# %pip install pandas
# %pip insatll numpy
# %pip install matplotlib
# %pip install seaborn
# %pip install scikit_learn

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, GridSearchCV, StratifiedKFold,
                                     cross_val_score, learning_curve, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, classification_report)

csv_path = 'synthetic_basketball_dataset.csv'   # change to your downloaded file name
df = pd.read_csv(csv_path)
print('Shape:', df.shape)
df.head()

Shape: (10000, 22)


,Match_ID,A_Off_Rating,A_Def_Rating,A_Net_Rating,A_Star_Player,A_Chemistry,A_Injury,A_Last5_WinPct,A_Momentum,A_Rest,...,B_Net_Rating,B_Star_Player,B_Chemistry,B_Injury,B_Last5_WinPct,B_Momentum,B_Rest,Home_Team,Tournament_Stage,Winner
0,0,113.973713,106.893886,7.079828,94.448975,0.899518,0.108755,0.593526,0.624098,3,...,-5.002731,76.325219,0.576689,0.363810,0.385038,0.382014,2,B,2,1
1,1,112.844411,111.336089,1.508322,87.113290,0.696632,0.139330,0.403514,0.457163,1,...,11.006888,101.985961,0.806116,0.084199,0.759965,0.756135,3,B,1,0
2,2,92.573326,99.648829,-7.075503,78.116044,0.514936,0.196338,0.139161,0.215553,2,...,0.787960,76.999023,0.618309,0.167988,0.357807,0.422560,2,B,0,0
3,3,115.422139,113.587522,1.834617,88.957537,0.753169,0.074773,0.547052,0.546526,2,...,-1.909460,81.294785,0.537236,0.369147,0.330580,0.379311,0,B,2,1
4,4,106.810893,107.513087,-0.702194,88.464371,0.532514,0.078207,0.645234,0.580122,0,...,-17.689937,58.560097,0.096696,0.174742,0.106837,0.122369,0,B,0,1


## Preprocessing

## Training

In [ ]:
# 1. Target encoding (needs exactly two classes)
classes = sorted(df[target_col].unique(), key=str)
if len(classes) != 2:
    raise ValueError(f'Winner has {len(classes)} unique values; need exactly 2: {classes[:10]}')
team_a_label, team_b_label = classes[0], classes[1]
print('1 (A wins) =', team_a_label, '| 0 (B wins) =', team_b_label)
y = (df[target_col] == team_a_label).astype(int).values

# 2. Drop unwanted columns
X = df.drop(columns=[target_col] + [c for c in DROP_COLS if c in df.columns])
for c in list(X.columns):
    if X[c].nunique() <= 1:
        print('Dropping constant column:', c)
        X = X.drop(columns=c)
    elif X[c].dtype == object and X[c].nunique() > MAX_UNIQUE_TEXT:
        print('Dropping high-cardinality text column:', c)
        X = X.drop(columns=c)

# 3. One-hot encode text columns, fill missing values
X = pd.get_dummies(X, drop_first=True).astype(float)
X = X.fillna(X.median()).fillna(0)
feature_names = list(X.columns)
X = X.values

# 4. Train/test split (80/20)
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[idx[:split]], X[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]
# X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

# 5. Standardize using training statistics, add bias column
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1
X_train = np.hstack([np.ones((X_train.shape[0], 1)), (X_train - mean) / std])
X_test = np.hstack([np.ones((X_test.shape[0], 1)), (X_test - mean) / std])

print()
print('Features after encoding:', len(feature_names))
print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Class balance (share of class 1) train:', round(y_train.mean(), 3), '| test:', round(y_test.mean(), 3))

In [ ]:
EPS = 1e-12

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, p):
    return -np.mean(y_true * np.log(p + EPS) + (1 - y_true) * np.log(1 - p + EPS))

w = np.zeros(X_train.shape[1])
lr = 0.1
epochs = 2000
lam = 0.01
m = len(y_train)
print('Hyperparameters: learning rate =', lr, '| epochs =', epochs, '| L2 lambda =', lam)

hist = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
for i in range(epochs):
    p = sigmoid(X_train @ w)
    grad = X_train.T @ (p - y_train) / m
    reg = lam * w / m
    reg[0] = 0
    w -= lr * (grad + reg)

    p_tr = sigmoid(X_train @ w)
    p_te = sigmoid(X_test @ w)
    hist['train_loss'].append(log_loss(y_train, p_tr))
    hist['test_loss'].append(log_loss(y_test, p_te))
    hist['train_acc'].append(np.mean((p_tr >= 0.5).astype(int) == y_train))
    hist['test_acc'].append(np.mean((p_te >= 0.5).astype(int) == y_test))
    if i % 200 == 0:
        tl = hist['train_loss'][-1]
        ta = hist['train_acc'][-1]
        print(f'Epoch {i:4d}  train loss = {tl:.4f}  train acc = {ta:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist['train_loss'], label='Train')
axes[0].plot(hist['test_loss'], label='Test')
axes[0].set_title('Log-loss vs epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[1].plot(hist['train_acc'], label='Train')
axes[1].plot(hist['test_acc'], label='Test')
axes[1].set_title('Accuracy vs epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()

## Predict

## Evaluation

In [ ]:
tr_acc, te_acc = m_train['Accuracy'], m_test['Accuracy']
gap = tr_acc - te_acc
print(f'Train accuracy = {tr_acc:.3f}, Test accuracy = {te_acc:.3f}, gap = {gap:.3f}')
print(f'Test baseline (always predict majority class) = {baseline_test:.3f}')
print()

if te_acc - baseline_test < 0.02:
    print('- The model barely beats the majority-class baseline: the features carry little signal about the winner.')
else:
    print(f'- The model beats the baseline by {te_acc - baseline_test:.3f}, so the features carry real signal.')

if gap > 0.05:
    print('- Train accuracy is clearly higher than test: signs of overfitting. Try stronger regularization (larger lam) or fewer features.')
elif tr_acc < 0.6:
    print('- Both accuracies are low: signs of underfitting. A non-linear model or better features may help.')
else:
    print('- Train and test accuracy are close: the model generalizes well.')

if te_acc > 0.98:
    print('- Accuracy is suspiciously high: check for leakage columns (final scores, point differences) and add them to DROP_COLS.')

print(f'- Test ROC AUC = {auc_test:.3f} (0.5 = random ranking, 1.0 = perfect ranking).')
prec_te, rec_te = m_test['Precision'], m_test['Recall']
balance = 'balanced between the two classes.' if abs(prec_te - rec_te) < 0.1 else 'the model favors one class over the other.'
print(f'- Test precision = {prec_te:.3f}, recall = {rec_te:.3f}: ' + balance)

# Feature influence (features are standardized, so weights are comparable)
weights = pd.Series(w[1:], index=feature_names)
top = weights.reindex(weights.abs().sort_values(ascending=False).index)[:10]
print()
print('Top features by absolute weight (positive weight raises the chance that A wins):')
display(top.round(4).to_frame('weight'))

top.sort_values().plot.barh(figsize=(8, 5), title='Top 10 feature weights')
plt.xlabel('Weight (standardized features)')
plt.tight_layout()
plt.show()